# Day 3: Parameterization of Monomers and Polymer Chain Generation

This notebook demonstrates how to parameterize monomers and generate polymer chains using **AmberTools**. It includes reusable functions to simplify the process for new molecules.

We will also visualize the molecules using **nglview**.

## Step 0: Initialization

This step sets up the environment for the tutorial. It installs required Python packages, creates necessary directories, and defines reusable functions.

**Important:** Run this step only once when starting the notebook or after restarting the kernel.

In [1]:
# --- Lightweight "reset" for clean re-run ---
from IPython.display import clear_output
import importlib, os, sys

clear_output(wait=True)
importlib.invalidate_caches()

# Install required packages
!{sys.executable} -m pip install nglview parmed > /dev/null

import nglview as nv
import parmed as pmd

# Set fixed project root
if "PROJECT_ROOT" not in os.environ:
    os.environ["PROJECT_ROOT"] = os.path.abspath(os.getcwd())

PROJECT_ROOT = os.environ["PROJECT_ROOT"]
print("Project root:", PROJECT_ROOT)

# Create subdirectories
os.makedirs(os.path.join(PROJECT_ROOT, "monomers"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, "polymers"), exist_ok=True)

# Define reusable functions
def run_command(command):
    """Runs a shell command and prints the output."""
    print(f"Executing: {command}")
    os.system(command)

def get_monomer_dir(mol_name):
    """Returns the directory path for a given monomer."""
    monomer_dir = os.path.join(PROJECT_ROOT, "monomers", mol_name)
    os.makedirs(monomer_dir, exist_ok=True)
    return monomer_dir

def visualize_molecule(file_path):
    """
    Visualize a molecule from any supported file (PDB, MOL2, etc.) 
    with orthographic camera and fully opaque atoms (no fading).
    
    Parameters:
        file_path (str): Path to the molecular structure file.
    """
    view = nv.show_file(file_path)
     # Orthographic camera
    view.camera = 'orthographic'
    # Disable fog and make atoms fully opaque
    view.stage.set_parameters(fog=False)
    for rep in view.representations:
        rep.update_params(opacity=1.0)
    
    return view

def visualize_monomer(mol_name):
    """
    Visualize a monomer from its MOL2 file with atom indices, 
    using an orthographic camera and no fading.

    Parameters:
        mol_name (str): Name of the monomer (assumes sqm.pdb file exists in monomer directory).
    """
    mol_path = f"{get_monomer_dir(mol_name)}/sqm.pdb"
    view = nv.show_file(mol_path)
    view.add_label(radius=2, color='black',label_type='atomname')
     # Orthographic camera
    view.camera = 'orthographic'
    # Disable fog and make atoms fully opaque
    view.stage.set_parameters(fog=False)
    for rep in view.representations:
        rep.update_params(opacity=1.0)
    return view


Project root: /Users/harish/Documents/Work_folder/MetaChem-workshops/polymermd-workshop/Day3-gen-polymer-solved


## Step 1: Prepare Monomer from SMILES or File

This step converts a SMILES string, PDB, or XYZ file into a 3D molecular structure in MOL2 format and visualizes the result.

### Parameters:
- `mol_name`: Name of the molecule.
- `input_type`: Type of input file (`smiles`, `pdb`, or `xyz`).
- `input_data`: SMILES string or path to the input file.

In [2]:
def prepare_monomer(mol_name, input_type, input_data):
    """
    Prepares a monomer by converting SMILES or file (PDB/XYZ/MOL2) to MOL2 format.
    Warnings from OpenBabel are suppressed.

    Parameters:
        mol_name (str): Name of the molecule.
        input_type (str): 'smiles' or 'file'.
        input_data (str): SMILES string or path to input file.
    """
    monomer_dir = get_monomer_dir(mol_name)
    os.makedirs(monomer_dir, exist_ok=True)
    os.chdir(monomer_dir)
    mol2_file = f"{mol_name}.mol2"
    
    if input_type.lower() == "smiles":
        cmd = f'obabel -:"{input_data}" -O {mol2_file} --gen3d 2>/dev/null'
    elif input_type.lower() == "file":
        cmd = f'obabel {input_data} -O {mol2_file} --gen3d 2>/dev/null'
    else:
        raise ValueError("Invalid input_type. Must be 'smiles' or 'file'.")
    
    run_command(cmd)
    print(f"Monomer {mol_name} prepared successfully in {monomer_dir}.")

# Example usage: prepare PEO from SMILES
mol_name = "PEO"
prepare_monomer(mol_name, input_type="smiles", input_data="CCOCCOCCOCCOCCOCC")
visualize_molecule(f"{get_monomer_dir(mol_name)}/{mol_name}.mol2")

Executing: obabel -:"CCOCCOCCOCCOCCOCC" -O PEO.mol2 --gen3d 2>/dev/null
Monomer PEO prepared successfully in /Users/harish/Documents/Work_folder/MetaChem-workshops/polymermd-workshop/Day3-gen-polymer-solved/monomers/PEO.


NGLWidget()

In [3]:
# Example usage: prepare PEO from SMILES
mol_name = "PET"
prepare_monomer(mol_name, input_type="file", input_data="PET_monomer.mol2")
visualize_molecule(f"{get_monomer_dir(mol_name)}/{mol_name}.mol2")

Executing: obabel PET_monomer.mol2 -O PET.mol2 --gen3d 2>/dev/null
Monomer PET prepared successfully in /Users/harish/Documents/Work_folder/MetaChem-workshops/polymermd-workshop/Day3-gen-polymer-solved/monomers/PET.


NGLWidget()

In [6]:
mol_name = "PE-test"
prepare_monomer(mol_name, input_type="file", input_data="PE-test.pdb")
visualize_molecule(f"{get_monomer_dir(mol_name)}/{mol_name}.mol2")

Executing: obabel PE-test.pdb -O PE-test.mol2 --gen3d 2>/dev/null
Monomer PE-test prepared successfully in /Users/harish/Documents/Work_folder/MetaChem-workshops/polymermd-workshop/Day3-gen-polymer-solved/monomers/PE-test.


NGLWidget()

## Step 2: Parameterize and Visualize Monomer

This step uses `antechamber` to assign GAFF atom types and AM1-BCC charges to the monomer.

### Parameters:
- `mol_name`: Name of the molecule.

In [7]:
def parameterize_monomer(mol_name):
    """
    Parameterizes a monomer using antechamber.
    
    Parameters:
        mol_name (str): Name of the molecule (without file extension).
    """
    monomer_dir = get_monomer_dir(mol_name)
    os.chdir(monomer_dir)
    run_command(f"antechamber -i {mol_name}.mol2 -fi mol2 -o {mol_name}_gaff.ac -fo ac -at gaff2 -an y -c bcc -nc 0 -rn {mol_name} -s 2 -pf y")
    print(f"Monomer {mol_name} parameterized successfully.")

# Parameterize the monomer
parameterize_monomer(mol_name="PET")

#Visualize the parameterized monomer
visualize_monomer(mol_name="PET")

Executing: antechamber -i PET.mol2 -fi mol2 -o PET_gaff.ac -fo ac -at gaff2 -an y -c bcc -nc 0 -rn PET -s 2 -pf y
Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to gaff2; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for mol2 File --
   Status: pass
Info: Finished reading file (PET.mol2); atoms read (30), bonds read (30).
Info: Determining atomic numbers from atomic symbols which are case sensitive.
Running: /Users/harish/miniconda3/envs/polymer_md/bin/bondtype -j full -i ANTECHAMBER_BOND_TYPE.AC0 -o ANTECHAMBER_BOND_TYPE.AC -f ac

Running: /Users/harish/miniconda3/envs/polymer_md/bin/atomtype -i ANTECHAMBER_AC.AC0 -o ANTECHAMBER_AC.AC -p gaff2

Running: /Users/harish/miniconda3/envs/polymer_md/bin/sqm -O -i sqm.in -o sqm.out

Running: /Users/harish/miniconda3/envs/polymer_md/bin/am1bcc -i ANTECHAMBER_AM1BCC_PRE.AC -o ANTECHAMBER_AM1BCC.AC -f ac -t bcc -p /Users/harish/mi

NGLWidget()

In [8]:
# Parameterize the monomer
parameterize_monomer(mol_name="PET")

#Visualize the parameterized monomer
visualize_monomer(mol_name="PET")

Executing: antechamber -i PET.mol2 -fi mol2 -o PET_gaff.ac -fo ac -at gaff2 -an y -c bcc -nc 0 -rn PET -s 2 -pf y
Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to gaff2; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for mol2 File --
   Status: pass
Info: Finished reading file (PET.mol2); atoms read (30), bonds read (30).
Info: Determining atomic numbers from atomic symbols which are case sensitive.
Running: /Users/harish/miniconda3/envs/polymer_md/bin/bondtype -j full -i ANTECHAMBER_BOND_TYPE.AC0 -o ANTECHAMBER_BOND_TYPE.AC -f ac

Running: /Users/harish/miniconda3/envs/polymer_md/bin/atomtype -i ANTECHAMBER_AC.AC0 -o ANTECHAMBER_AC.AC -p gaff2

Running: /Users/harish/miniconda3/envs/polymer_md/bin/sqm -O -i sqm.in -o sqm.out

Running: /Users/harish/miniconda3/envs/polymer_md/bin/am1bcc -i ANTECHAMBER_AM1BCC_PRE.AC -o ANTECHAMBER_AM1BCC.AC -f ac -t bcc -p /Users/harish/mi

NGLWidget()

## Step 3: Define Chain, Head, and Tail

This step defines the chain, head, and tail of the monomer for polymerization.

### Parameters:
- `mol_name`: Name of the molecule.
- `head_id`, `tail_id`: Atom indices for the head and tail.
- `head_omit`, `tail_omit`: Atom indices to omit near the head and tail.

In [4]:
def define_chain_head_tail(mol_name, head_id, tail_id, head_omit, tail_omit, chain_id="A"):
    """
    Defines the chain, head, and tail of the monomer and saves the files.
    Also ensures a valid chain ID for tleap (replaces illegal 'X').
    
    Parameters:
        mol_name (str): Name of the molecule (without file extension).
        head_id (int): Atom index for the head.
        tail_id (int): Atom index for the tail.
        head_omit (list): Atom indices to omit near the head.
        tail_omit (list): Atom indices to omit near the tail.
        chain_id (str): Single-character chain ID (default 'A') for tleap.
    """
    monomer_dir = get_monomer_dir(mol_name)
    os.chdir(monomer_dir)
    chain_file = f"{mol_name}.chain"
    head_file = f"{mol_name}.head"
    tail_file = f"{mol_name}.tail"

    # Write chain/head/tail definition files
    with open(chain_file, "w") as chain, open(head_file, "w") as head, open(tail_file, "w") as tail:
        # Chain file
        chain.write(f"HEAD_NAME {head_id}\n")
        chain.write(f"TAIL_NAME {tail_id}\n")
        for omit in head_omit + tail_omit:
            chain.write(f"OMIT_NAME {omit}\n")
        chain.write(f"PRE_HEAD_TYPE {tail_id}\n")
        chain.write(f"POST_TAIL_TYPE {head_id}\n")
        chain.write("CHARGE 0\n")
        
        # Head file
        head.write(f"HEAD_NAME {head_id}\n")
        head.write(f"TAIL_NAME {tail_id}\n")
        for omit in tail_omit:
            head.write(f"OMIT_NAME {omit}\n")
        head.write(f"POST_TAIL_TYPE {head_id}\n")
        head.write("CHARGE 0\n")
        
        # Tail file
        tail.write(f"HEAD_NAME {head_id}\n")
        tail.write(f"TAIL_NAME {tail_id}\n")
        for omit in head_omit:
            tail.write(f"OMIT_NAME {omit}\n")
        tail.write(f"PRE_HEAD_TYPE {tail_id}\n")
        tail.write("CHARGE 0\n")

    # Run prepgen to generate .prepi files
    run_command(f"prepgen -i {mol_name}_gaff.ac -o {mol_name}.prepi -f prepi -m {mol_name}.chain -rn {mol_name} -rf {mol_name}.res")
    run_command(f"prepgen -i {mol_name}_gaff.ac -o HPT.prepi -f prepi -m {mol_name}.head -rn HPT -rf HPT.res")
    run_command(f"prepgen -i {mol_name}_gaff.ac -o TPT.prepi -f prepi -m {mol_name}.tail -rn TPT -rf TPT.res")

    print(f"Chain, head, and tail files for {mol_name} saved successfully with chain ID '{chain_id}'.")

# Define chain, head, and tail
#define_chain_head_tail(mol_name="PEO", head_id=1, tail_id=15, head_omit=[0, 17, 18, 19], tail_omit=[16, 40, 41, 42])

#define_chain_head_tail(mol_name="PEO", 
#                       head_id='C1', 
#                       tail_id='C10', 
#                       head_omit=['C', 'H', 'H1', 'H2'], 
#                       tail_omit=['C11','H23','H24','H25'])

In [10]:
define_chain_head_tail(mol_name="PEO", 
                       head_id='C1', # head_id is the Atom label where the Head end group is connected
                       tail_id='C10', # tail_id is the Atom label where the Tail end group is connected
                       head_omit=['C', 'H', 'H1', 'H2'], # head_omit is the group of atoms containing HEAD end group
                       tail_omit=['C11','H23','H24','H25']) # tail_omit is the group of atoms containing TAIL end group

Executing: prepgen -i PEO_gaff.ac -o PEO.prepi -f prepi -m PEO.chain -rn PEO -rf PEO.res

PRE_HEAD_TYPE is   C10
POST_TAIL_TYPE is    C1
Net charge of truncated molecule is     0.00
HEAD_ATOM      2   C1
TAIL_ATOM     16  C10
MAIN_CHAIN     1    2   C1
MAIN_CHAIN     2   16  C10
OMIT_ATOM      1    1    C
OMIT_ATOM      2   18    H
OMIT_ATOM      3   19   H1
OMIT_ATOM      4   20   H2
OMIT_ATOM      5   17  C11
OMIT_ATOM      6   41  H23
OMIT_ATOM      7   42  H24
OMIT_ATOM      8   43  H25
Number of mainchain atoms (including head and tail atom):     2
Number of omited atoms:     8Executing: prepgen -i PEO_gaff.ac -o HPT.prepi -f prepi -m PEO.head -rn HPT -rf HPT.res

POST_TAIL_TYPE is    C1
Net charge of truncated molecule is     0.00
HEAD_ATOM      2   C1
TAIL_ATOM     16  C10
MAIN_CHAIN     1    2   C1
MAIN_CHAIN     2   16  C10
OMIT_ATOM      1   17  C11
OMIT_ATOM      2   41  H23
OMIT_ATOM      3   42  H24
OMIT_ATOM      4   43  H25
Number of mainchain atoms (including head and t

In [11]:
visualize_monomer(mol_name="PET")

NGLWidget()

In [12]:
define_chain_head_tail(mol_name="PET", 
                       head_id='C9', # head_id is the Atom label where the Head end group is connected
                       tail_id='C12', # tail_id is the Atom label where the Tail end group is connected
                       head_omit=['H5'], # head_omit is the group of atoms containing HEAD end group
                       tail_omit=['H12']) # tail_omit is the group of atoms containing TAIL end group

Executing: prepgen -i PET_gaff.ac -o PET.prepi -f prepi -m PET.chain -rn PET -rf PET.res

PRE_HEAD_TYPE is   C12
POST_TAIL_TYPE is    C9
Net charge of truncated molecule is     0.00
HEAD_ATOM     11   C9
TAIL_ATOM     16  C12
MAIN_CHAIN     1   11   C9
MAIN_CHAIN     2   16  C12
OMIT_ATOM      1   21   H5
OMIT_ATOM      2   28  H12
Number of mainchain atoms (including head and tail atom):     2
Number of omited atoms:     2Executing: prepgen -i PET_gaff.ac -o HPT.prepi -f prepi -m PET.head -rn HPT -rf HPT.res

POST_TAIL_TYPE is    C9
Net charge of truncated molecule is     0.00
HEAD_ATOM     11   C9
TAIL_ATOM     16  C12
MAIN_CHAIN     1   11   C9
MAIN_CHAIN     2   16  C12
OMIT_ATOM      1   28  H12
Number of mainchain atoms (including head and tail atom):     2
Number of omited atoms:     1Executing: prepgen -i PET_gaff.ac -o TPT.prepi -f prepi -m PET.tail -rn TPT -rf TPT.res

PRE_HEAD_TYPE is   C12
Net charge of truncated molecule is     0.00
HEAD_ATOM     11   C9
TAIL_ATOM     16  

## Step 4: Generate Polymer Chain

This step builds a polymer chain using the defined monomer, head, and tail.

### Parameters:
- `n_mono_repeat`: Number of monomers in a repeat unit.
- `n_mono_pol`: Total number of monomers in the polymer.

In [5]:
import os
import random

def generate_polymer_tleap(
    mol_names,
    n_mono_repeat,
    n_mono_pol,
    copolymer_type="homo",
    head_group=None,
    tail_group=None
):
    """
    Generate polymer structures using AMBER tleap from parameterised
    repeat units.

    This function constructs homopolymers, block copolymers, or random
    copolymers by assembling *parameterised repeat units* (PREPI files)
    and writes the corresponding tleap input, PDB, and AMBER topology files.

    ----------------------------
    PARAMETERS
    ----------------------------
    mol_names : list[str]
        Names of monomer repeat units.
        Each name must correspond to:
            {get_monomer_dir(mol_name)}/{mol_name}.prepi

    n_mono_repeat : list[int]
        Number of monomer units represented by ONE parameterised
        repeat unit for each mol_name.
        Example:
            PET repeat unit may represent 2 monomers.

    n_mono_pol : list[int]
        Desired total number of monomer units in the final polymer
        for each mol_name.

        Constraint:
            n_mono_pol[i] must be divisible by n_mono_repeat[i].

    copolymer_type : str, optional
        Polymer architecture:
            "homo"   → homopolymer (single mol_name)
            "block"  → block copolymer (ordered by mol_names)
            "random" → random copolymer (shuffled repeat units)

        Default: "homo"

    head_group : str, optional
        Monomer name whose directory contains HPT.prepi.
        If None, defaults to mol_names[0].

        Loaded once as:
            loadamberprep {get_monomer_dir(head_group)}/HPT.prepi

    tail_group : str, optional
        Monomer name whose directory contains TPT.prepi.
        If None, defaults to mol_names[-1].

        Loaded once as:
            loadamberprep {get_monomer_dir(tail_group)}/TPT.prepi

    ----------------------------
    RETURNS
    ----------------------------
    polymer_tag : str
        Unique identifier for the polymer, e.g.:
            "PEO40_PET20"

        Used to name output directory and files.

    ----------------------------
    FILES GENERATED
    ----------------------------
    polymers/{polymer_tag}/
        ├── tleap.in
        ├── tleap.out
        ├── polymer_<N>mer.pdb
        ├── polymer_<N>mer.prmtop
        └── polymer_<N>mer.inpcrd
    """

    # ----------------------------
    # Input validation
    # ----------------------------
    if not (len(mol_names) == len(n_mono_repeat) == len(n_mono_pol)):
        raise ValueError(
            "mol_names, n_mono_repeat, and n_mono_pol must have the same length"
        )

    # Default head and tail assignment
    if head_group is None:
        head_group = mol_names[0]
    if tail_group is None:
        tail_group = mol_names[-1]

    # ----------------------------
    # Determine number of repeat units
    # ----------------------------
    # tleap 'sequence' works in terms of parameterised repeat units,
    # NOT raw monomer counts.
    repeat_units = {}

    for mol, n_rep, n_pol in zip(mol_names, n_mono_repeat, n_mono_pol):
        if n_pol % n_rep != 0:
            raise ValueError(
                f"{mol}: n_mono_pol ({n_pol}) must be divisible by "
                f"n_mono_repeat ({n_rep})"
            )
        repeat_units[mol] = n_pol // n_rep

    # ----------------------------
    # Build polymer sequence
    # ----------------------------
    # The sequence contains repeat-unit names understood by tleap.
    if len(mol_names) == 1:
        # Homopolymer
        core_sequence = [mol_names[0]] * repeat_units[mol_names[0]]

    else:
        if copolymer_type == "block":
            # Ordered blocks: PEO...PET...
            core_sequence = []
            for mol in mol_names:
                core_sequence.extend([mol] * repeat_units[mol])

        elif copolymer_type == "random":
            # Random distribution of repeat units
            core_sequence = []
            for mol, n_units in repeat_units.items():
                core_sequence.extend([mol] * n_units)
            random.shuffle(core_sequence)

        else:
            raise ValueError(
                "copolymer_type must be 'homo', 'block', or 'random'"
            )

    # Convert sequence list to tleap-compatible string
    sequence = " ".join(core_sequence)

    # Total monomer count (used for file naming)
    total_monomers = sum(n_mono_pol)

    # ----------------------------
    # Polymer tag and directory
    # ----------------------------
    polymer_tag = "_".join(
        f"{mol}{n}" for mol, n in zip(mol_names, n_mono_pol)
    )
    polymer_name = f"{polymer_tag}_{copolymer_type}"
    polymer_dir = os.path.join(PROJECT_ROOT, "polymers", polymer_name)
    os.makedirs(polymer_dir, exist_ok=True)

    print(f"[Polymer Name] {polymer_name}")
    print(f"[Polymer directory] {polymer_dir}")

    # ----------------------------
    # Construct tleap input
    # ----------------------------
    prep_lines = set()

    # Load repeat-unit PREPI files
    for mol in mol_names:
        prep_lines.add(
            f"loadamberprep {get_monomer_dir(mol)}/{mol}.prepi"
        )

    # Load head and tail capping groups ONCE
    prep_lines.add(
        f"loadamberprep {get_monomer_dir(head_group)}/HPT.prepi"
    )
    prep_lines.add(
        f"loadamberprep {get_monomer_dir(tail_group)}/TPT.prepi"
    )

    prep_block = "\n".join(sorted(prep_lines))

    # Final tleap input file
    tleap_input = f"""
source leaprc.gaff2
{prep_block}

mol = sequence {{{sequence}}}

savepdb mol {polymer_dir}/polymer_{total_monomers}mer.pdb
saveamberparm mol {polymer_dir}/polymer_{total_monomers}mer.prmtop \
                 {polymer_dir}/polymer_{total_monomers}mer.inpcrd
quit
"""

    tleap_input_path = os.path.join(polymer_dir, "tleap.in")
    with open(tleap_input_path, "w") as f:
        f.write(tleap_input)

    # ----------------------------
    # Run tleap
    # ----------------------------
    print(f"Running tleap for {polymer_name}")
    os.system(f"tleap -s -f {tleap_input_path} > {polymer_dir}/tleap.out")
    os.system(f"tail -n 10 {polymer_dir}/tleap.out")

    # Return polymer identifier for downstream workflows
    return polymer_name

### Homo Polymers 

In [6]:
# ============================================================
# Generate a PEO homopolymer using parameterised repeat units
# ============================================================
#
# This cell constructs a PEO homopolymer using AMBER tleap.
# The polymer is built from *parameterised repeat units* (.prepi),
# not individual atoms or single-monomer fragments.
#
# ------------------------------------------------------------
# Parameter description
# ------------------------------------------------------------
#
# mol_names = ["PEO"]
#   - Defines the monomer (repeat unit) used in the polymer.
#   - Requires the file:
#       {get_monomer_dir("PEO")}/PEO.prepi
#   - A single entry implies a homopolymer.
#
# n_mono_repeat = [5]
#   - Each parameterised PEO repeat unit represents 5 chemical
#     monomer units.
#
# n_mono_pol = [25]
#   - Desired total number of monomer units in the final polymer.
#   - Number of repeat units used by tleap:
#       25 / 5 = 5 repeat units
#
# copolymer_type = "homo"
#   - Specifies homopolymer architecture.
#   - Included for clarity and consistency.
#
# head_group = "PEO"
#   - Head capping group (HPT.prepi) is loaded from:
#       {get_monomer_dir("PEO")}/HPT.prepi
#
# tail_group = "PEO"
#   - Tail capping group (TPT.prepi) is loaded from:
#       {get_monomer_dir("PEO")}/TPT.prepi
#
# ------------------------------------------------------------
# Files generated
# ------------------------------------------------------------
# polymers/PEO25/
#   ├── tleap.in
#   ├── tleap.out
#   ├── polymer_25mer.pdb
#   ├── polymer_25mer.prmtop
#   └── polymer_25mer.inpcrd
#
# ------------------------------------------------------------
# Returned value
# ------------------------------------------------------------
# polymer_name = "PEO25"
#   - Unique polymer identifier for downstream workflows
#     (e.g., GROMACS, Moltemplate, batch simulations)
#
# ============================================================

polymer_name = generate_polymer_tleap(
    mol_names=["PET"],
    n_mono_repeat=[1],
    n_mono_pol=[25],
    copolymer_type="homo",
    head_group="PET",
    tail_group="PET"
)

print("Generated polymer:", polymer_name)

[Polymer Name] PET25_homo
[Polymer directory] /Users/harish/Documents/Work_folder/MetaChem-workshops/polymermd-workshop/Day3-gen-polymer-solved/polymers/PET25_homo
Running tleap for PET25_homo
 total 150 improper torsions applied
 200 improper torsions in old prep form
Building H-Bond parameters.
Incorporating Non-Bonded adjustments.
Not Marking per-residue atom chain types.
Marking per-residue atom chain types.
 (no restraints)
	Quit

Exiting LEaP: Errors = 0; Warnings = 0; Notes = 0.
Generated polymer: PET25_homo


In [7]:
visualize_molecule(f"{PROJECT_ROOT}/polymers/{polymer_name}/polymer_25mer.pdb")

NGLWidget()

### Block co-polymers

In [21]:
# ============================================================
# Generate a PEO–PET block copolymer using parameterised repeats
# ============================================================
#
# This cell constructs a BLOCK copolymer consisting of a PEO block
# followed by a PET block using AMBER tleap.
#
# The polymer is assembled from parameterised repeat units (.prepi),
# ensuring chemical consistency with the force-field definition.
#
# ------------------------------------------------------------
# Parameter description
# ------------------------------------------------------------
#
# mol_names = ["PEO", "PET"]
#   - Defines the order of monomer blocks in the polymer.
#   - Required PREPI files:
#       {get_monomer_dir("PEO")}/PEO.prepi
#       {get_monomer_dir("PET")}/PET.prepi
#
# n_mono_repeat = [5, 1]
#   - PEO: one parameterised repeat unit represents 5 monomer units
#   - PET: one parameterised repeat unit represents 1 monomer unit
#
# n_mono_pol = [25, 25]
#   - Desired total monomer count in the polymer:
#       PEO: 25 monomers → 25 / 5 = 5 repeat units
#       PET: 25 monomers → 25 / 1 = 25 repeat units
#
# copolymer_type = "block"
#   - Polymer architecture is block-ordered:
#       PEO block → PET block
#
# head_group = "PEO"
#   - Head capping group (HPT.prepi) loaded from:
#       {get_monomer_dir("PEO")}/HPT.prepi
#
# tail_group = "PET"
#   - Tail capping group (TPT.prepi) loaded from:
#       {get_monomer_dir("PET")}/TPT.prepi
#
# ------------------------------------------------------------
# Resulting polymer composition
# ------------------------------------------------------------
# Sequence (repeat units):
#   PEO × 5  →  PET × 25
#
# Total monomers:
#   25 (PEO) + 25 (PET) = 50 monomers
#
# ------------------------------------------------------------
# Files generated
# ------------------------------------------------------------
# polymers/PEO25_PET25/
#   ├── tleap.in
#   ├── tleap.out
#   ├── polymer_50mer.pdb
#   ├── polymer_50mer.prmtop
#   └── polymer_50mer.inpcrd
#
# ------------------------------------------------------------
# Returned value
# ------------------------------------------------------------
# polymer_name = "PEO25_PET25"
#
# ============================================================

polymer_name = generate_polymer_tleap(
    mol_names=["PEO", "PET"],
    n_mono_repeat=[5, 1],
    n_mono_pol=[25, 25],
    copolymer_type="block",
    head_group="PEO",
    tail_group="PET"
)

print("Generated polymer:", polymer_name)

[Polymer Name] PEO25_PET25_block
[Polymer directory] /Users/harish/Documents/Work_folder/MetaChem-workshops/polymermd-workshop/Day3-gen-polymer-solved/polymers/PEO25_PET25_block
Running tleap for PEO25_PET25_block
 total 150 improper torsions applied
 200 improper torsions in old prep form
Building H-Bond parameters.
Incorporating Non-Bonded adjustments.
Not Marking per-residue atom chain types.
Marking per-residue atom chain types.
 (no restraints)
	Quit

Exiting LEaP: Errors = 0; Warnings = 0; Notes = 0.
Generated polymer: PEO25_PET25_block


In [23]:
polymer_name

'PEO25_PET25_block'

In [24]:
visualize_molecule(f"{PROJECT_ROOT}/polymers/{polymer_name}/polymer_50mer.pdb")

NGLWidget()

### Random co-polymers

In [25]:
# ============================================================
# Generate a PEO–PET RANDOM copolymer using parameterised repeats
# ============================================================
#
# This cell constructs a RANDOM copolymer where PEO and PET repeat
# units are distributed randomly along the polymer backbone.
#
# The total monomer count and repeat-unit definitions are preserved,
# but the sequence order is randomly shuffled.
#
# ------------------------------------------------------------
# Parameter description
# ------------------------------------------------------------
#
# mol_names = ["PEO", "PET"]
#   - Defines the monomer types present in the polymer.
#
# n_mono_repeat = [5, 1]
#   - PEO repeat unit represents 5 monomers
#   - PET repeat unit represents 1 monomer
#
# n_mono_pol = [25, 25]
#   - Desired total monomer count:
#       PEO → 5 repeat units
#       PET → 25 repeat units
#
# copolymer_type = "random"
#   - Repeat units are randomly distributed along the chain.
#   - Stoichiometry is preserved, order is not.
#
# head_group = "PEO"
#   - Head capping group (HPT.prepi) loaded from:
#       {get_monomer_dir("PEO")}/HPT.prepi
#
# tail_group = "PET"
#   - Tail capping group (TPT.prepi) loaded from:
#       {get_monomer_dir("PET")}/TPT.prepi
#
# ------------------------------------------------------------
# Resulting polymer composition
# ------------------------------------------------------------
# Total monomers:
#   25 (PEO) + 25 (PET) = 50 monomers
#
# Sequence:
#   Random permutation of:
#       PEO × 5  +  PET × 25
#
# ------------------------------------------------------------
# Returned value
# ------------------------------------------------------------
# polymer_name = "PEO25_PET25"
#
# ============================================================

polymer_name = generate_polymer_tleap(
    mol_names=["PEO", "PET"],
    n_mono_repeat=[5, 1],
    n_mono_pol=[25, 25],
    copolymer_type="random",
    head_group="PEO",
    tail_group="PET"
)

print("Generated polymer:", polymer_name)

[Polymer Name] PEO25_PET25_random
[Polymer directory] /Users/harish/Documents/Work_folder/MetaChem-workshops/polymermd-workshop/Day3-gen-polymer-solved/polymers/PEO25_PET25_random
Running tleap for PEO25_PET25_random
 total 150 improper torsions applied
 200 improper torsions in old prep form
Building H-Bond parameters.
Incorporating Non-Bonded adjustments.
Not Marking per-residue atom chain types.
Marking per-residue atom chain types.
 (no restraints)
	Quit

Exiting LEaP: Errors = 0; Warnings = 0; Notes = 0.
Generated polymer: PEO25_PET25_random


In [26]:
visualize_molecule(f"{PROJECT_ROOT}/polymers/{polymer_name}/polymer_50mer.pdb")

NGLWidget()

## Step 5: Convert to GROMACS Formats and Visualize Polymer

This step converts the AMBER topology and coordinate files into GROMACS-compatible formats.

In [8]:
# ------------------------------------------------------------
# Convert polymer generated by tleap to GROMACS format
# ------------------------------------------------------------

# polymer_name is the tag returned by generate_polymer_tleap,
# e.g., "PEO25" or "PEO25_PET25"
# polymer_dir is automatically set in tleap function
# (or you can recompute it if needed)

polymer_name = 'PEO25_PET25_random'  # Use the last generated polymer
n_mono_pol = 50  # Total monomer counts used in the last generation
polymer_dir = os.path.join(PROJECT_ROOT, "polymers", polymer_name)

# Load the AMBER topology and coordinate files
import parmed as pmd

amber = pmd.load_file(
    f"{polymer_dir}/polymer_{str(n_mono_pol)}mer.prmtop", 
    f"{polymer_dir}/polymer_{str(n_mono_pol)}mer.inpcrd"
)

# Save in GROMACS formats
amber.save(f"{polymer_dir}/polymer_{str(n_mono_pol)}mer.gro", overwrite=True)
amber.save(f"{polymer_dir}/topol.top", format="gromacs", overwrite=True)

print(f"Conversion completed: .gro and .top files generated for polymer '{polymer_name}'.")

Conversion completed: .gro and .top files generated for polymer 'PEO25_PET25_random'.


In [9]:
# Visualize the polymer chain
view = visualize_molecule(f"{polymer_dir}/polymer_{str(n_mono_pol)}mer.gro")
view

NGLWidget()